# Stream the chat endpoint over HTTP with `httpx`

Sends a real HTTP request to a **running** `chat-service` and streams the response token by token, the way a client would. Unlike `test_chat.ipynb` (which drives the app in-process via `TestClient`), this goes over the wire to `localhost:8000`.

Start the server first (in `backend/`):

```bash
uv run uvicorn chat_service.asgi:app --reload
```

Then run the cells below.

In [ ]:
import json
import uuid

import httpx

BASE_URL = "http://localhost:8000"
CHAT_URL = f"{BASE_URL}/api/v1/chat"

# Sanity check the server is up.
try:
    ping = httpx.get(f"{BASE_URL}/ping", timeout=5.0)
    print(f"GET /ping -> {ping.status_code} {ping.json()}")
except Exception as e:
    print(f"Server not reachable at {BASE_URL}: {type(e).__name__}: {e}")
    print("Start it with:  uv run uvicorn chat_service.asgi:app --reload")

# Register a fresh user; every request authenticates with the returned API key
# via the Authorization: Bearer header (the key is shown once, at registration).
email = f"demo-{uuid.uuid4().hex[:8]}@example.com"
reg = httpx.post(f"{BASE_URL}/api/v1/users", json={"email": email}, timeout=10.0)
reg.raise_for_status()
api_key = reg.json()["api_key"]
HEADERS = {"Authorization": f"Bearer {api_key}"}
print(f"registered {email}; using API key {api_key[:8]}... as Bearer token")


def consume_stream(resp) -> None:
    """Print tokens from an NDJSON chat stream; surface error events."""
    for line in resp.iter_lines():
        if not line:
            continue
        event = json.loads(line)
        if event["type"] == "token":
            print(event["content"], end="", flush=True)
        elif event["type"] == "error":
            print(f"\n[error] {event['content']}")
    print()

GET /ping -> 200 {'message': 'pong'}
registered demo-f9189d29@example.com; using API key wrt_SIWy... as Bearer token


## Stream a single request

`httpx.stream` keeps the connection open and yields the body as it arrives. We print each chunk as it comes in and capture the `X-Session-Id` header the server returns.

In [2]:
session_id = None
with httpx.stream(
    "POST",
    CHAT_URL,
    headers=HEADERS,
    json={"question": "Please write a limerick about Stockholm"},
    timeout=60.0,
) as resp:
    resp.raise_for_status()
    session_id = resp.headers.get("X-Session-Id")
    print(f"status: {resp.status_code}  |  X-Session-Id: {session_id}")
    print("--- streamed answer ---")
    consume_stream(resp)
print()

status: 200  |  X-Session-Id: 97da02d6-1bd6-4f23-af2d-405b88e6ce27
--- streamed answer ---
There once was a city, Stockholm,  
Where islands and bridges all welcome.  
With waters that gleam,  
Like some Nordic dream,  
It charms every heart that has come.



## Continue the same session

Echo the `X-Session-Id` back as `session_id` to continue the conversation. (History isn't persisted yet, so the token only round-trips for now — this shows the client-side pattern.)

In [3]:
with httpx.stream(
    "POST",
    CHAT_URL,
    headers=HEADERS,
    json={"question": "Now write the same limerick about another city", "session_id": session_id},
    timeout=60.0,
) as resp:
    resp.raise_for_status()
    print(f"echoed X-Session-Id matches: {resp.headers.get('X-Session-Id') == session_id}")
    print("--- streamed answer ---")
    consume_stream(resp)
print()

echoed X-Session-Id matches: True
--- streamed answer ---
There once was a city, Berlin,  
Where history and nightlife both spin.  
With art on display,  
And culture each day,  
It draws you right gladly within.



In [4]:
with httpx.stream(
    "POST",
    CHAT_URL,
    headers=HEADERS,
    json={"question": "Now give me a recounting of our full conversation", "session_id": session_id},
    timeout=60.0,
) as resp:
    resp.raise_for_status()
    print(f"echoed X-Session-Id matches: {resp.headers.get('X-Session-Id') == session_id}")
    print("--- streamed answer ---")
    consume_stream(resp)
print()

echoed X-Session-Id matches: True
--- streamed answer ---
Here’s a full recounting of our conversation:

1. You asked: “Please write a limerick about Stockholm”
2. I replied with this limerick:

   There once was a city, Stockholm,  
   Where islands and bridges all welcome.  
   With waters that gleam,  
   Like some Nordic dream,  
   It charms every heart that has come.

3. You then asked: “Now write the same limerick about another city”
4. I replied with a similar limerick about Berlin:

   There once was a city, Berlin,  
   Where history and nightlife both spin.  
   With art on display,  
   And culture each day,  
   It draws you right gladly within.

5. You then asked: “Now give me a recounting of our full conversation”

That brings us to the current message.



## Async variant

The same request with `httpx.AsyncClient` + `aiter_lines`, for use inside async code. Jupyter supports top-level `await`.

In [4]:
async with httpx.AsyncClient(timeout=60.0) as client:
    async with client.stream(
        "POST",
        CHAT_URL,
        headers=HEADERS,
        json={"question": "Say hello in exactly three words."},
    ) as resp:
        resp.raise_for_status()
        print(f"X-Session-Id: {resp.headers.get('X-Session-Id')}")
        print("--- streamed answer ---")
        async for line in resp.aiter_lines():
            if not line:
                continue
            event = json.loads(line)
            if event["type"] == "token":
                print(event["content"], end="", flush=True)
            elif event["type"] == "error":
                print(f"\n[error] {event['content']}")
print()

X-Session-Id: 6cb1ec72-4b10-4a59-bcca-d0f3191f35ca
--- streamed answer ---
Hello to you


## List your conversations

Fetch every conversation owned by the authenticated user (the registered API key), newest first. The streaming turns above created one, so it should show up here.

In [5]:
resp = httpx.get(f"{BASE_URL}/api/v1/conversations", headers=HEADERS, timeout=10.0)
resp.raise_for_status()
conversations = resp.json()["conversations"]
print(f"{len(conversations)} conversation(s) for {email}:")
for c in conversations:
    marker = "   <- current session" if c["session_id"] == session_id else ""
    print(f"  {c['created_at']}  {c['session_id']}{marker}")

1 conversation(s) for demo-f9189d29@example.com:
  2026-06-11T20:32:06.492324Z  97da02d6-1bd6-4f23-af2d-405b88e6ce27   <- current session


## Fetch a conversation's messages

Returns the full transcript for one conversation — `conversation_id` is the session token (mapped internally to the checkpointer `thread_id`). Roles come back as `user` / `assistant`.

In [5]:
resp = httpx.get(f"{BASE_URL}/api/v1/conversations/{session_id}/messages", headers=HEADERS, timeout=10.0)
resp.raise_for_status()
messages = resp.json()["messages"]
print(f"transcript for {session_id} ({len(messages)} message(s)):\n")
for m in messages:
    print(f"[{m['role']}] {m['content']}")
    print("-" * 60)

transcript for 663786b6-31ac-4dee-a71b-2da7a3d37d93 (4 message(s)):

[user] Please write a limerick about Stockholm
------------------------------------------------------------
[assistant] There once was a city, Stockholm,  
Whose islands made visitors calm.  
With bridges so bright,  
And midsummer light,  
It charmed every heart with its balm.
------------------------------------------------------------
[user] Now write the same limerick about another city
------------------------------------------------------------
[assistant] There once was a city, Kyoto,  
Where temples and blossoms did glow.  
With lanterns at night,  
And gardens in light,  
It charmed every heart as they’d go.
------------------------------------------------------------


## Ownership is enforced

A different user (a different registered API key) cannot read this conversation — the API returns 404, the same status it would for a non-existent one, so existence isn't leaked.

In [7]:
# Register a *different* user and try to read the first user's conversation.
other_email = f"other-{uuid.uuid4().hex[:8]}@example.com"
other_reg = httpx.post(f"{BASE_URL}/api/v1/users", json={"email": other_email}, timeout=10.0)
other_reg.raise_for_status()
other_headers = {"Authorization": f"Bearer {other_reg.json()['api_key']}"}
other = httpx.get(
    f"{BASE_URL}/api/v1/conversations/{session_id}/messages",
    headers=other_headers,
    timeout=10.0,
)
print(f"as another user -> HTTP {other.status_code} (expected 404)")

as 'someone-else' -> HTTP 404 (expected 404)
